# silver_cadastro

Camada **Silver** do cadastro: `unidades_consumidoras` e `medidores`.

- **Entrada:** `bronze.tb_unidades_consumidoras`, `bronze.tb_medidores`
- **Saída:** `silver.tb_unidades_consumidoras`, `silver.tb_medidores` e as quarentenas (`_quarentena`)
- **Segue o Padrão de Limpeza:** cada tabela passa pelos 9 passos (ler → estrutura → tipos → domínios → duplicatas → consistência → integridade → checagens → gravar).
- **Todas as outras Silver dependem deste notebook.** Não mude nome nem tipo de coluna sem combinar com o time.
- Regras levantadas no notebook `exploracao_cadastro`.

In [0]:
%run ./funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
NOTEBOOK = "silver_cadastro"

In [0]:
# ---------- Unidades consumidoras ----------

# Identificador válido: "UC" + 6 dígitos. Fora disso = registro de teste -> quarentena
ID_UC_PADRAO = r"^UC\d{6}$"

# Domínios (o mapear_dominio já resolve caixa e acento; aqui só as grafias diferentes)
MAPA_UF = {"PERNAMBUCO": "PE", "P.E.": "PE"}
CLASSES = ["RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "RURAL", "PODER PUBLICO"]
SUBGRUPOS = ["B1", "B2", "B3", "A4"]
STATUS_UC = ["ATIVA"]

# CEP: além dos textos de "sem valor" padrão, -1 e a data-sentinela também significam "sem CEP"
VALORES_NULOS_CEP = VALORES_NULOS + ("-1", "9999-12-31")
CEP_PADRAO = r"^5[0-6]\d{6}$"          # 8 dígitos na faixa de Pernambuco (50 a 56)

# Ligação antes desta data = preenchimento (sempre 1º/jan; vazio de 1950 a 1989)
DATA_LIGACAO_MINIMA = "1950-01-01"

# Consistências (hoje 100% respeitadas; violação vira flag)
CLASSE_DOCUMENTO = {"RESIDENCIAL": 11, "RURAL": 11,                    # CPF
                    "COMERCIAL": 14, "INDUSTRIAL": 14, "PODER PUBLICO": 14}   # CNPJ
CLASSE_SUBGRUPO = {"RESIDENCIAL": "B1", "COMERCIAL": "B3", "INDUSTRIAL": "A4",
                   "RURAL": "B2", "PODER PUBLICO": "B3"}
ALIMENTADOR_MUNICIPIO = {
    "REC": "RECIFE", "JAB": "JABOATÃO DOS GUARARAPES", "OLI": "OLINDA",
    "PAU": "PAULISTA", "CAB": "CABO DE SANTO AGOSTINHO", "CAM": "CAMARAGIBE",
    "SAO": "SÃO LOURENÇO DA MATA", "IGA": "IGARASSU", "IPO": "IPOJUCA",
    "ABR": "ABREU E LIMA", "MOR": "MORENO", "ITA": "ITAPISSUMA",
    "ILH": "ILHA DE ITAMARACÁ", "ARA": "ARAÇOIABA",
}

# Textos de exibição: maiúsculas mantendo acento (+ correção de encoding)
TEXTOS_EXIBICAO = ["nome_titular", "logradouro", "bairro", "municipio"]

In [0]:
# ---------- Medidores ----------

# Domínios
MAPA_TIPO_MEDIDOR = {"ELETRO-MECANICO": "ELETROMECANICO", "SMART METER": "SMART"}
TIPOS_MEDIDOR = ["ELETROMECANICO", "ELETRONICO", "SMART"]
STATUS_MEDIDOR = ["ATIVO", "RETIRADO"]

# Colunas de data (4 formatos na origem; o parse_data trata todos)
DATAS_MEDIDOR = ["data_instalacao", "data_retirada", "data_ultima_manutencao"]

# Consistência: cada modelo pertence sempre ao mesmo tipo (hoje 100%)
MODELO_TIPO = {
    "CG-500": "ELETROMECANICO", "EM-1200": "ELETROMECANICO", "EM-1400": "ELETROMECANICO",
    "EL-2200": "ELETRONICO",    "EL-3000": "ELETRONICO",     "NX-700": "ELETRONICO",
    "SM-4100": "SMART",         "SM-4500": "SMART",          "IQ-900": "SMART",
}

## Parte A: Unidades consumidoras

In [0]:
uc = ler_bronze(CATALOGO, "unidades_consumidoras")

N_UC_BRONZE = uc.count()
REMOVIDAS_UC = {}      # o que cada passo remover entra aqui, com a quantidade medida

print(f"Bronze: {N_UC_BRONZE:,} linhas")

In [0]:
from functools import reduce

# 3a. Cabeçalho do CSV repetido no meio do arquivo
uc = remover_cabecalho_repetido(uc)
REMOVIDAS_UC["cabecalho"] = N_UC_BRONZE - uc.count()
registrar("unidades_consumidoras", "cabeçalho do CSV repetido no meio do arquivo",
          "remover_cabecalho_repetido", "REMOVIDO", REMOVIDAS_UC["cabecalho"])

# 3b. Espaços nas pontas (e texto só de espaços -> nulo)
COLUNAS_UC = [c for c in uc.columns if not c.startswith("_")]

n_espacos = contar_espacos_nas_pontas(uc)

uc = aparar_textos(uc)
registrar("unidades_consumidoras", "espaços no início/fim dos textos (logradouro, cep)",
          "aparar_textos", "CORRIGIDO", n_espacos)

In [0]:
# 3c. Encoding quebrado: texto UTF-8 que foi lido como Latin-1 (ex.: "JordÃ£o" -> "Jordão")
MOJIBAKE = r"[ÃÂ][\x{80}-\x{BF}]"
SO_MOJIBAKE = r"^([\x{00}-\x{7F}]|[ÃÂ][\x{80}-\x{BF}])*$"

def corrigir_encoding(col):
    """Desfaz a dupla codificação: volta o texto para bytes Latin-1 e relê como UTF-8.
    Só age em texto que tem o padrão quebrado e nenhum outro caractere especial."""
    c = F.col(col)
    reverter = c.rlike(MOJIBAKE) & c.rlike(SO_MOJIBAKE)
    return F.when(reverter, F.decode(F.encode(c, "ISO-8859-1"), "UTF-8")).otherwise(c)

tem_mojibake = reduce(lambda a, b: a | b, [F.col(c).rlike(MOJIBAKE) for c in TEXTOS_EXIBICAO])
n_encoding = uc.filter(tem_mojibake).count()

uc = uc.withColumns({c: corrigir_encoding(c) for c in TEXTOS_EXIBICAO})

n_restante = uc.filter(tem_mojibake).count()
assert n_restante == 0, f"{n_restante} linhas continuam com encoding quebrado"
registrar("unidades_consumidoras", "encoding quebrado (UTF-8 lido como Latin-1) nos textos de exibição",
          "corrigir_encoding (local)", "CORRIGIDO", n_encoding)

In [0]:
# 4a. Datas e coordenadas: confere que nada se perde antes de converter
checar_parse(uc, ["data_ligacao"], parse_data, "unidades_consumidoras")
checar_parse(uc, ["latitude", "longitude"], parse_decimal, "unidades_consumidoras", precisao=9, escala=6)

uc = uc.withColumns({
    "data_ligacao": parse_data("data_ligacao"),
    "latitude":     parse_decimal("latitude", 9, 6),
    "longitude":    parse_decimal("longitude", 9, 6),
})

# 4b. CPF/CNPJ: só dígitos (tira a máscara)
n_mascara = uc.filter(F.col("cpf_cnpj").rlike(r"\D")).count()
uc = uc.withColumn("cpf_cnpj", somente_digitos("cpf_cnpj"))
registrar("unidades_consumidoras", "CPF/CNPJ com máscara (pontos, barra, hífen)",
          "somente_digitos", "CORRIGIDO", n_mascara)

In [0]:
# 5a. UF: 5 grafias -> PE
n_uf = uc.filter(F.col("uf") != "PE").count()
uc = uc.withColumn("uf", mapear_dominio("uf", MAPA_UF))
registrar("unidades_consumidoras", "UF em 5 grafias (pe, Pernambuco, P.E. ...)",
          "mapear_dominio(MAPA_UF)", "CORRIGIDO", n_uf)

# 5b. Classe: 15 grafias -> 5 classes (só padronizar já resolve: mapa vazio)
n_classe = uc.filter(~F.col("classe").isin(CLASSES)).count()
uc = uc.withColumn("classe", mapear_dominio("classe", {}))
registrar("unidades_consumidoras", "classe em 15 grafias (caixa e acento)",
          "mapear_dominio (só padronização)", "CORRIGIDO", n_classe)

# 5c. Domínios que hoje já vêm limpos: padronizamos mesmo assim, como proteção
uc = uc.withColumns({c: padronizar_texto(c) for c in ["subgrupo_tarifario", "status", "alimentador"]})

# 5d. Textos de exibição: maiúsculas, mantendo acento, sem espaços duplos
muda_texto = reduce(lambda a, b: a | b,
                    [~F.col(c).eqNullSafe(padronizar_texto(c, remover_acentos=False))
                     for c in TEXTOS_EXIBICAO])
n_textos = uc.filter(muda_texto).count()
uc = uc.withColumns({c: padronizar_texto(c, remover_acentos=False) for c in TEXTOS_EXIBICAO})
registrar("unidades_consumidoras", "textos de exibição com caixa mista (nome, endereço, bairro, município)",
          "padronizar_texto(remover_acentos=False)", "CORRIGIDO", n_textos)

In [0]:
n_antes = uc.count()
uc = deduplicar(uc)
REMOVIDAS_UC["duplicatas"] = n_antes - uc.count()

registrar("unidades_consumidoras",
          "id_uc repetido (linhas idênticas ou diferentes só na caixa do nome e na máscara do CPF/CNPJ)",
          "padronizar + deduplicar(df)", "REMOVIDO", REMOVIDAS_UC["duplicatas"])
print(f"Antes: {n_antes:,} | depois: {n_antes - REMOVIDAS_UC['duplicatas']:,}")

In [0]:
# 7a. Registros de teste: id_uc fora do padrão "UC + 6 dígitos" -> quarentena
id_valido = F.coalesce(F.col("id_uc").rlike(ID_UC_PADRAO), F.lit(False))

uc_quarentena = (uc.filter(~id_valido)
                   .withColumn("motivo_quarentena",
                               F.lit("id_uc fora do padrão UC + 6 dígitos (registro de teste)")))
uc = uc.filter(id_valido)

n_teste = uc_quarentena.count()
registrar("unidades_consumidoras",
          "registros de teste (TESTE0000-TESTE0199, CPF 00000000000, nome de preenchimento)",
          "id_uc fora de ^UC\\d{6}$", "QUARENTENA", n_teste)

# 7b. CEP: textos de "sem valor" e valores fora da faixa de PE -> nulo
cep_limpo = somente_digitos(valores_para_nulo("cep", VALORES_NULOS_CEP))
cep_final = F.when(cep_limpo.rlike(CEP_PADRAO), cep_limpo)

n_cep = uc.filter(F.col("cep").isNotNull() & cep_final.isNull()).count()
uc = uc.withColumn("cep", cep_final)
registrar("unidades_consumidoras",
          "CEP sem valor escrito como N/A, NULL, null, -, -1 ou 9999-12-31",
          "valores_para_nulo + somente_digitos + faixa 50-56 (8 dígitos)", "CORRIGIDO", n_cep)

print(f"UCs: {uc.count():,} | quarentena: {n_teste:,} | CEPs nulos: {uc.filter('cep is null').count():,}")

In [0]:
# 7c. Data de ligação antes de 1950: flag + data vira nulo
suspeita = F.col("data_ligacao") < F.to_date(F.lit(DATA_LIGACAO_MINIMA))
uc = criar_flag(uc, "flag_data_ligacao_suspeita", suspeita)
uc = uc.withColumn("data_ligacao",
                   F.when(F.col("flag_data_ligacao_suspeita"), None).otherwise(F.col("data_ligacao")))

# 7d. Consistências de negócio (hoje 100%): violação vira flag
def mapa_spark(d):
    """Transforma um dicionário Python num mapa do Spark, para consultar por coluna."""
    return F.create_map([F.lit(x) for par in d.items() for x in par])

prefixo_alimentador = F.substring("alimentador", 5, 3)        # "ALM-REC-008" -> "REC"
uc = criar_flag(uc, "flag_documento_classe_inconsistente",
                ~F.length("cpf_cnpj").eqNullSafe(mapa_spark(CLASSE_DOCUMENTO)[F.col("classe")]))
uc = criar_flag(uc, "flag_classe_subgrupo_inconsistente",
                ~F.col("subgrupo_tarifario").eqNullSafe(mapa_spark(CLASSE_SUBGRUPO)[F.col("classe")]))
uc = criar_flag(uc, "flag_alimentador_municipio_inconsistente",
                ~F.col("municipio").eqNullSafe(mapa_spark(ALIMENTADOR_MUNICIPIO)[prefixo_alimentador]))

# Conta as 4 flags numa única leitura e registra cada uma
FLAGS_UC = {
    "flag_data_ligacao_suspeita":
        ("data de ligação antes de 1950 (sempre 1º/jan; vazio de 1950 a 1989)", "data < 1950-01-01 -> flag + data nula"),
    "flag_documento_classe_inconsistente":
        ("classe incompatível com o documento (CPF x CNPJ)", "CLASSE_DOCUMENTO"),
    "flag_classe_subgrupo_inconsistente":
        ("subgrupo tarifário incompatível com a classe", "CLASSE_SUBGRUPO"),
    "flag_alimentador_municipio_inconsistente":
        ("alimentador de outro município", "ALIMENTADOR_MUNICIPIO"),
}

registrar_flags("unidades_consumidoras", uc, FLAGS_UC)

In [0]:
# Passo 8 (integridade): não se aplica. Unidades é a tabela de referência das outras.

# Passo 9: checagens. Se qualquer uma falhar, o notebook para ANTES de gravar.
checar_unicidade(uc, ["id_uc"], "unidades_consumidoras")

checar_sem_nulos(uc, ["id_uc", "cpf_cnpj", "nome_titular", "logradouro", "numero", "bairro",
                      "municipio", "uf", "classe", "subgrupo_tarifario", "alimentador",
                      "status", "latitude", "longitude"], "unidades_consumidoras")

checar_dominio(uc, "uf", ["PE"], "unidades_consumidoras")
checar_dominio(uc, "classe", CLASSES, "unidades_consumidoras")
checar_dominio(uc, "subgrupo_tarifario", SUBGRUPOS, "unidades_consumidoras")
checar_dominio(uc, "status", STATUS_UC, "unidades_consumidoras")
checar_dominio(uc, "municipio", list(ALIMENTADOR_MUNICIPIO.values()), "unidades_consumidoras")

reconciliar("unidades_consumidoras", N_UC_BRONZE, uc.count(), uc_quarentena.count(), REMOVIDAS_UC)

In [0]:
# Ordem final das colunas: negócio -> flags -> rastreio (é o "contrato" com o time)
uc = uc.select(*COLUNAS_UC, *FLAGS_UC, "_source_file", "_ingestion_timestamp")

n_uc = gravar_silver(uc, CATALOGO, "unidades_consumidoras", quarentena=uc_quarentena)

spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras").printSchema()

## Parte B: Medidores

In [0]:
# Passo 2: ler e contar
md = ler_bronze(CATALOGO, "medidores")
N_MD_BRONZE = md.count()
REMOVIDAS_MD = {}
print(f"Bronze: {N_MD_BRONZE:,} linhas")

# Passo 3: estrutura
md = remover_cabecalho_repetido(md)                  # hoje não há cabeçalho repetido: proteção
REMOVIDAS_MD["cabecalho"] = N_MD_BRONZE - md.count()
if REMOVIDAS_MD["cabecalho"]:
    registrar("medidores", "cabeçalho do CSV repetido no meio do arquivo",
              "remover_cabecalho_repetido", "REMOVIDO", REMOVIDAS_MD["cabecalho"])

COLUNAS_MD = [c for c in md.columns if not c.startswith("_")]

n_espacos_md = contar_espacos_nas_pontas(md)

md = aparar_textos(md)
registrar("medidores", "espaços no início/fim do modelo", "aparar_textos", "CORRIGIDO", n_espacos_md)

In [0]:
# Passo 4: tipos (3 datas em 4 formatos + sentinela 9999-12-31)
n_formato = md.filter(F.col("data_instalacao").isNotNull()
                      & ~F.col("data_instalacao").rlike(r"^\d{4}-\d{2}-\d{2}$")).count()
n_sentinela = md.filter(F.col("data_retirada") == "9999-12-31").count()

checar_parse(md, DATAS_MEDIDOR, parse_data, "medidores")
md = md.withColumns({c: parse_data(c) for c in DATAS_MEDIDOR})

registrar("medidores", "data de instalação em 4 formatos (yyyy-MM-dd, MM-dd-yyyy, dd/MM/yyyy, dd.MM.yyyy)",
          "parse_data", "CORRIGIDO", n_formato)
registrar("medidores", "data de retirada 9999-12-31 significando 'sem retirada'",
          "parse_data (sentinela -> nulo)", "CORRIGIDO", n_sentinela)

# Passo 5: domínios
n_tipo = md.filter(~F.col("tipo").isin(TIPOS_MEDIDOR)).count()
md = md.withColumn("tipo", mapear_dominio("tipo", MAPA_TIPO_MEDIDOR))
registrar("medidores", "tipo em 10 grafias (caixa, acento, hífen, 'smart meter')",
          "mapear_dominio(MAPA_TIPO_MEDIDOR)", "CORRIGIDO", n_tipo)

md = md.withColumns({c: padronizar_texto(c) for c in ["modelo", "status"]})   # proteção

In [0]:
n_antes = md.count()
md = deduplicar(md)
REMOVIDAS_MD["duplicatas"] = n_antes - md.count()

registrar("medidores", "linhas duplicadas idênticas", "deduplicar(df)", "REMOVIDO",
          REMOVIDAS_MD["duplicatas"])
print(f"Antes: {n_antes:,} | depois: {n_antes - REMOVIDAS_MD['duplicatas']:,}")

In [0]:
ret, ins = F.col("data_retirada"), F.col("data_instalacao")

md = criar_flag(md, "flag_status_retirada_inconsistente",
                ((F.col("status") == "ATIVO") & ret.isNotNull())
                | ((F.col("status") == "RETIRADO") & ret.isNull()))
md = criar_flag(md, "flag_retirada_antes_instalacao", ret < ins)
md = criar_flag(md, "flag_modelo_tipo_inconsistente",
                ~F.col("tipo").eqNullSafe(mapa_spark(MODELO_TIPO)[F.col("modelo")]))

FLAGS_MD = {
    "flag_status_retirada_inconsistente":
        ("status incompatível com a data de retirada (ATIVO com retirada ou RETIRADO sem)", "status x data_retirada"),
    "flag_retirada_antes_instalacao":
        ("data de retirada anterior à instalação", "data_retirada < data_instalacao"),
    "flag_modelo_tipo_inconsistente":
        ("tipo incompatível com o modelo (ou modelo desconhecido)", "MODELO_TIPO"),
}

registrar_flags("medidores", md, FLAGS_MD)

n_manutencao_nula = md.filter(F.col("data_ultima_manutencao").isNull()).count()
registrar("medidores", "data_ultima_manutencao nunca preenchida na origem",
          "coluna mantida, toda nula", "MANTIDO", n_manutencao_nula)

In [0]:
uc_silver = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")

md, md_quarentena = separar_orfaos(md, uc_silver, "id_uc",
                                   motivo="id_uc inexistente em silver.tb_unidades_consumidoras")
n_orfaos = md_quarentena.count()
registrar("medidores", "medidor de UC inexistente no cadastro", "separar_orfaos vs silver.tb_unidades_consumidoras",
          "QUARENTENA", n_orfaos)

In [0]:
checar_unicidade(md, ["id_medidor"], "medidores")

checar_sem_nulos(md, ["id_medidor", "id_uc", "modelo", "tipo", "status", "data_instalacao"], "medidores")

checar_dominio(md, "tipo", TIPOS_MEDIDOR, "medidores")
checar_dominio(md, "status", STATUS_MEDIDOR, "medidores")

# Garantia para o silver_medicao: no máximo 1 medidor ATIVO por UC
checar_unicidade(md.filter(F.col("status") == "ATIVO"), ["id_uc"], "medidores ATIVOS por UC")

reconciliar("medidores", N_MD_BRONZE, md.count(), md_quarentena.count(), REMOVIDAS_MD)

In [0]:
md = md.select(*COLUNAS_MD, *FLAGS_MD, "_source_file", "_ingestion_timestamp")

n_md = gravar_silver(md, CATALOGO, "medidores", quarentena=md_quarentena)

spark.table(f"{CATALOGO}.silver.tb_medidores").printSchema()

## Fechamento: log de qualidade

In [0]:
gravar_log(CATALOGO, NOTEBOOK)

display(spark.table(f"{CATALOGO}.silver.tb_log_qualidade")
             .filter(F.col("notebook") == NOTEBOOK)
             .orderBy("tabela", "destino"))